# Use saved models in `models/`

This notebook loads the pre-trained URL and image models stored in the repo’s `models/` folder and runs inference.

Notes:
- URL models are sklearn pipelines saved with `joblib` (`.joblib`).
- The image model is a PyTorch `state_dict` saved to `models/img_clf_model.pt` and requires `torch` at runtime.


In [3]:
# If running in a fresh environment, uncomment: 
# %pip install -r ../requirements.txt
# %pip install torch torchvision  # needed only for image inference

from pathlib import Path
import sys

# Allow imports from `src/` when running the notebook directly.
REPO_ROOT = Path.cwd().resolve().parent
sys.path.append(str(REPO_ROOT / 'src'))

MODEL_DIR = REPO_ROOT / 'models'
print('Repo root:', REPO_ROOT)
print('Models:', [p.name for p in MODEL_DIR.glob('*')])


Repo root: D:\Files\Github\ml-2-url-img-classifier
Models: ['img_clf_model.pt', 'url_clf_features_only.joblib', 'url_clf_w_embedding.joblib']


## URL inference (joblib pipelines)

The repo ships two URL models:
- `url_clf_features_only.joblib`: numeric URL features only
- `url_clf_w_embedding.joblib`: Word2Vec embedding + numeric features

Both can be used through `hf_phishing_url.inference.predict_urls`, which will rebuild the feature dataframe from raw URL strings.


In [4]:
from hf_phishing_url.inference import load_url_pipeline, predict_urls

urls = [
    'https://www.google.com/',
    'http://login.verify-account-security.example.com/update',
    'http://123.45.67.89/secure/login',
]

url_models = [
    MODEL_DIR / 'url_clf_features_only.joblib',
    MODEL_DIR / 'url_clf_w_embedding.joblib',
]

for mp in url_models:
    if not mp.exists():
        print('Missing:', mp)
        continue

    pipe = load_url_pipeline(mp)
    preds = predict_urls(pipe, urls, threshold=0.5)

    print('\nModel:', mp.name)
    for p in preds:
        print(f'{p.phishing_proba:.4f}\t{int(p.is_phishing)}\t{p.url}')



Model: url_clf_features_only.joblib
0.0208	0	https://www.google.com/
0.5881	1	http://login.verify-account-security.example.com/update
0.9904	1	http://123.45.67.89/secure/login

Model: url_clf_w_embedding.joblib
0.0395	0	https://www.google.com/
0.9984	1	http://login.verify-account-security.example.com/update
0.9991	1	http://123.45.67.89/secure/login


## Image inference (PyTorch weights)

This loads `models/img_clf_model.pt` (a `state_dict`) into the `SimpleCNN` architecture mirrored from `notebooks/train-image-model.ipynb`.

If you don’t have PyTorch installed, skip this section.


In [ ]:
from pathlib import Path

weights = MODEL_DIR / 'img_clf_model.pt'
print('Weights exist:', weights.exists())

# Set this to a real screenshot path on your machine.
IMAGE_PATH = Path('D:\\Downloads\\scam.jpg')

try:
    import torch  # noqa: F401
except Exception as e:
    print('PyTorch not available:', e)
else:
    from phishing_image.inference import load_image_model, predict_image

    if not weights.exists():
        raise FileNotFoundError(weights)
    if not IMAGE_PATH.exists():
        raise FileNotFoundError(IMAGE_PATH)

    model = load_image_model(weights, device=None, image_size=224)
    pred = predict_image(model, IMAGE_PATH, threshold=0.5, image_size=224)
    pred


Weights exist: True


FileNotFoundError: path\to\screenshot.jpg

## (Optional) CLI equivalent

You can run the same inference from a terminal:

- URL: `python scripts/predict.py url --model models/url_clf_features_only.joblib "https://example.com/login"`
- URL (embedding): `python scripts/predict.py url --model models/url_clf_w_embedding.joblib "https://example.com/login"`
- Image: `python scripts/predict.py image --weights models/img_clf_model.pt path\\to\\screenshot.jpg`
